# Tutorial 04 - Project: Monthly Sales Trend & Forecasting Engine

Generates 12-month synthetic sales records for four products, computes
growth indicators, and plots seasonal patterns.


---
## Step 0: Portable Environment Bootstrap (Local IDE / Jupyter + Google Colab)

Locates the `APDS/07_Monthly_Sales_Forecasting/` project folder (anchored on `utils.py`; the generated CSVs go to the project's git-ignored `data/` folder) by walking upward from the
current working directory — so the notebook runs identically from the repository
root, the project folder or its `notebooks/` sub-folder. On Google Colab the
repository is cloned into `/content/APSD_TAI_LAB` automatically (opening the
notebook directly from GitHub also works). The resolved project folder is added
to `sys.path` so its local modules (e.g. `utils`, `students`) import cleanly.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

PROJECT_NAME = "07_Monthly_Sales_Forecasting"
ANCHOR_REL = "utils.py"  # file that uniquely identifies the project folder
REPO_URL = "https://github.com/Vtheonly/APSD_TAI_LAB.git"
IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ

if IN_COLAB:
    # Google Colab: clone the repository (unless already present, e.g. on Drive)
    _repo = Path("/content/APSD_TAI_LAB")
    if not (_repo / "APDS").is_dir():
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(_repo)], check=True
        )
    PROJ_DIR = _repo / "APDS" / PROJECT_NAME
else:
    # Local IDE / Jupyter: find the project folder by walking upward from the
    # working directory (works from the repo root, the project folder or its
    # notebooks/ sub-folder, regardless of where Jupyter was launched).
    _cwd = Path.cwd().resolve()
    PROJ_DIR = next(
        (
            c / "APDS" / PROJECT_NAME
            for c in (_cwd, *_cwd.parents)
            if (c / "APDS" / PROJECT_NAME / ANCHOR_REL).exists()
        ),
        None,
    )
    if PROJ_DIR is None:
        raise FileNotFoundError(
            f"Project folder APDS/{PROJECT_NAME} (anchor: {ANCHOR_REL}) not found "
            f"above {_cwd}. Launch Jupyter from inside the cloned repository."
        )
if str(PROJ_DIR) not in sys.path:
    sys.path.insert(0, str(PROJ_DIR))

print(f"Project folder : {PROJ_DIR}")
print(f"Google Colab   : {IN_COLAB}")


### 1. Generate the Monthly Dataset


In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from utils import generate_random_sales

np.random.seed(42)
os.makedirs(PROJ_DIR / "data", exist_ok=True)

# Generate 12 monthly dates for 2025
months = pd.date_range(start="2025-01-01", end="2025-12-01", freq="MS")
num_m = len(months)

df_initial = pd.DataFrame({
    "Date": months,
    "Product_A": generate_random_sales(50, 100, num_m),
    "Product_B": generate_random_sales(30, 80, num_m),
    "Product_C": generate_random_sales(20, 60, num_m),
    "Product_D": generate_random_sales(10, 50, num_m),
})

df_initial.to_csv(PROJ_DIR / "data" / "initial.csv", index=False)
display(df_initial.head())


### 2. Metric Calculations & Quarter Mapping


In [ ]:
df_final = df_initial.copy()
df_final["Month"] = df_final["Date"].dt.strftime("%b")
pcols = ["Product_A", "Product_B", "Product_C", "Product_D"]

df_final["Total_Sales"] = df_final[pcols].sum(axis=1)
df_final["Average_Sales"] = df_final[pcols].mean(axis=1)
df_final["MoM_Growth"] = df_final["Total_Sales"].pct_change() * 100

def assign_quarter(dt):
    m = dt.month
    if m <= 3: return "Q1"
    elif m <= 6: return "Q2"
    elif m <= 9: return "Q3"
    else: return "Q4"

df_final["Quarter"] = df_final["Date"].apply(assign_quarter)
df_final["Max_Product"] = df_final[pcols].idxmax(axis=1)
df_final["Min_Product"] = df_final[pcols].idxmin(axis=1)

df_final.to_csv(PROJ_DIR / "data" / "final.csv", index=False)
display(df_final.head())


### 3. Quarterly Summary Table


In [ ]:
pivot_q = df_final.pivot_table(index="Quarter", values=pcols + ["Total_Sales"], aggfunc="mean")
pivot_q.to_csv(PROJ_DIR / "data" / "output.csv")
display(pivot_q)


### 4. 4-Panel Visualization Grid


In [ ]:
sns.set_theme(style="whitegrid")
fig, axs = plt.subplots(2, 2, figsize=(15, 10))

# 1. Product Trends
for col in pcols:
    axs[0, 0].plot(df_final["Month"], df_final[col], marker="o", label=col)
axs[0, 0].set_title("Product Monthly Trends", fontweight="bold")
axs[0, 0].legend()

# 2. Stacked Sales Volume
df_final.set_index("Month")[pcols].plot(kind="bar", stacked=True, ax=axs[0, 1], colormap="viridis")
axs[0, 1].set_title("Monthly Total Sales Volume", fontweight="bold")

# 3. Density Heatmap
sns.heatmap(df_final.set_index("Month")[pcols], annot=True, fmt="d", cmap="YlGnBu", ax=axs[1, 0])
axs[1, 0].set_title("Monthly Sales Density", fontweight="bold")

# 4. Variance Boxplot (single colour -> no deprecation warning on seaborn >= 0.13)
sns.boxplot(data=df_final[pcols], color="lightgreen", ax=axs[1, 1])
axs[1, 1].set_title("Product Sales Distributions", fontweight="bold")

plt.tight_layout()
plt.show()
